<a href="https://colab.research.google.com/github/acoiman/pdt/blob/main/asthma_mortality/notebooks/Python/07.Asthma_Mortality_PD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 👨‍👩‍👧‍👦 Population Density (PD)

In this notebook we will  calculate the **population density (PD)**  for each department in Argentinafrom 2001 to 2022 based on our  annual population estimates.

##📦 Import Required Libraries

In [ ]:
# geospatial data handling
import geopandas as gpd

# other libraries
import pandas as pd
import os
from itables import init_notebook_mode

In [ ]:
# Set the PROJ_LIB path
os.environ['PROJ_LIB'] = "/opt/conda/envs/gds/share/proj"

In [ ]:
# change directory to work folder (at the begining,  docker container enter into /home/jovyan/)
%cd work

In [ ]:
 # Pandas Copy-on-Write
 pd.options.mode.copy_on_write = True

##🗂 Load Dataset

In [ ]:
# Load dataset with data NAMR + PM2.5 + PM10 and burned areas per department
gdf = gpd.read_file("pdt/asthma_mortality/data/gpkg/tma_pm25_pm10_ba_2001_2022.gpkg")

In [ ]:
# visualiz the geodataframe
init_notebook_mode(all_interactive=True)
gdf.head()

## 👪 Calculate Population Density

In [ ]:
def add_population_density(gdf):
    """
    Adds population density columns to a GeoDataFrame.

    Parameters:
    - gdf: GeoDataFrame in EPSG:4326 with population columns like 'A_2001', 'A_2002', ...

    Returns:
    - GeoDataFrame with new columns 'PD_2001', 'PD_2002', ... (people per km²)
    """
    # Project to EPSG:5347 POSGAR 2007 / Argentina 5
    gdf_proj = gdf.to_crs(epsg=5347)

    # Calculate area in square kilometers
    gdf_proj["area_km2"] = gdf_proj.geometry.area / 1e6

    # Identify population columns
    pop_cols = [col for col in gdf.columns if col.startswith("A_") and col[2:].isdigit()]

    # Add population density columns
    for col in pop_cols:
        year = col.split("_")[1]
        gdf_proj[f"PD_{year}"] = round(gdf_proj[col] / gdf_proj["area_km2"], 2)

    # Drop intermediate area column if not needed
    gdf_proj = gdf_proj.drop(columns=["area_km2"])

    # Reproject back to EPSG:4326
    gdf_final = gdf_proj.to_crs(epsg=4326)

    return gdf_final

### Apply Population Density Function


In [ ]:
# Add population density
gdf_with_density = add_population_density(gdf)

### Preview Resulting GeoDataFrame and save results

In [ ]:
# Display the first few rows of the DataFrame
init_notebook_mode(all_interactive=True)
gdf_with_density.head()

In [ ]:
# Save the GeoDataFrame to a GeopackeoPackage file
#gdf_with_density.to_file("pdt/asthma_mortality/data/gpkg/tma_pm25_pm10_ba_pd_2001_2022.gpkg", driver="GPKG")